# 1-Etiquetado de partes del discurso (POS)

In [1]:
import spacy
import pandas as pd

In [2]:
nlp = spacy.load("en_core_web_sm")
#Si es la primera vez que ejecutas esto o recibes el error "No se encuentra el modelo 'en_core_web_sm'",
# ejecuta lo siguiente en tu terminal: python -m spacy download en_core_web_sm

In [3]:
# Nuestro texto es de 'Emma' de Jane Austen
# Hemos eliminado la puntuación, las minúsculas, pero dejamos las palabras vacías
emma_ja = "Emma Woodhouse, guapa, inteligente y rica, con un hogar confortable y una disposición feliz, parecía reunir algunas de las mejores bendiciones de la existencia y había vivido casi veintiún años en el mundo con muy poco que la angustiara o molestara. Era la menor de las dos hijas de un padre muy cariñoso e indulgente y, como consecuencia del matrimonio de su hermana, había sido la dueña de su casa desde una época muy temprana. Su madre había muerto hacía demasiado tiempo como para que ella tuviera más que un recuerdo vago de sus caricias y su lugar había sido ocupado por una excelente mujer como institutriz que había sido poco menos que una madre en afecto. Dieciséis años había estado la señorita Taylor en la familia del señor Woodhouse menos como institutriz que como amiga. Tenía mucho cariño por ambas hijas, pero particularmente por Emma. Entre ellas era más la intimidad de hermanas, incluso antes de que la señorita Taylor dejara de tener el cargo nominal de institutriz. La suavidad de su temperamento apenas le había permitido imponer ninguna restricción y, como la sombra de la autoridad había desaparecido hacía tiempo, habían estado viviendo juntas como amigas, muy unidas mutuamente y Emma. Haciendo sólo lo que le gustaba, teniendo en alta estima el juicio de la señorita Taylor, pero dirigida principalmente por su propio criterio."
print(emma_ja)

Emma Woodhouse, guapa, inteligente y rica, con un hogar confortable y una disposición feliz, parecía reunir algunas de las mejores bendiciones de la existencia y había vivido casi veintiún años en el mundo con muy poco que la angustiara o molestara. Era la menor de las dos hijas de un padre muy cariñoso e indulgente y, como consecuencia del matrimonio de su hermana, había sido la dueña de su casa desde una época muy temprana. Su madre había muerto hacía demasiado tiempo como para que ella tuviera más que un recuerdo vago de sus caricias y su lugar había sido ocupado por una excelente mujer como institutriz que había sido poco menos que una madre en afecto. Dieciséis años había estado la señorita Taylor en la familia del señor Woodhouse menos como institutriz que como amiga. Tenía mucho cariño por ambas hijas, pero particularmente por Emma. Entre ellas era más la intimidad de hermanas, incluso antes de que la señorita Taylor dejara de tener el cargo nominal de institutriz. La suavidad d

In [4]:
# crea un documento espacial a partir de nuestro texto: esto generará tokens y sus etiquetas pos asociadas
spacy_doc = nlp(emma_ja)

In [5]:
# extrae los tokens y las etiquetas pos en un marco de datos
pos_df = pd.DataFrame(columns=["token", "pos_tag"])

In [6]:
for token in spacy_doc:
    pos_df = pd.concat([pos_df,
                        pd.DataFrame.from_records([{"token": token.text, "pos_tag": token.pos_}])],
                       ignore_index= True)

In [7]:
pos_df.head(10)

,token,pos_tag
0,Emma,PROPN
1,Woodhouse,PROPN
2,",",PUNCT
3,guapa,VERB
4,",",PUNCT
5,inteligente,PROPN
6,y,PROPN
7,rica,PROPN
8,",",PUNCT
9,con,PROPN


In [8]:
# recuento de frecuencia de tokens
pos_df_counts = pos_df.groupby(["token","pos_tag"]).size().reset_index(name= "counts").sort_values(by="counts", ascending= False)
pos_df_counts.head(15)

,token,pos_tag,counts
0,",",PUNCT,13
36,de,PROPN,11
1,.,PUNCT,8
119,que,PROPN,8
31,como,PROPN,7
83,la,PROPN,7
148,y,PROPN,7
130,su,PROPN,5
101,muy,PROPN,4
142,una,PROPN,4


In [9]:
# recuentos de pos_tags
pos_df_poscounts = pos_df_counts.groupby(["pos_tag"])["token"].count().sort_values(ascending= False)
pos_df_poscounts.head(15)

pos_tag
PROPN    107
NOUN      24
VERB       7
X          5
ADJ        3
PUNCT      2
AUX        1
INTJ       1
Name: token, dtype: int64

In [10]:
# ver sustantivos más comunes
nouns = pos_df_counts[pos_df_counts.pos_tag == "NOUN"][0:10]
nouns

,token,pos_tag,counts
76,institutriz,NOUN,2
98,mujer,NOUN,1
95,molestara,NOUN,1
96,mucho,NOUN,1
109,parecía,NOUN,1
127,sido,NOUN,1
138,tiempo,NOUN,1
133,sólo,NOUN,1
110,particularmente,NOUN,1
124,señor,NOUN,1


In [11]:
# ver verbos más comunes
verbs = pos_df_counts[pos_df_counts.pos_tag == "ADJ"][0:10]
verbs

,token,pos_tag,counts
126,sido,ADJ,1
147,viviendo,ADJ,1
2,Dieciséis,ADJ,1


# 2- Reconocimiento de entidades nombradas

In [12]:
import spacy
from spacy import displacy
from spacy import tokenizer
import re
nlp = spacy.load('en_core_web_sm')

In [13]:
google_text = "Google fue fundada el 4 de septiembre de 1998 por los informáticos Larry Page y Sergey Brin mientras estudiaban doctorado en la Universidad de Stanford, California. Juntos, poseen aproximadamente el 14 % de sus acciones cotizadas en bolsa y controlan el 56 % del poder de voto de sus accionistas mediante acciones con supervoto. La compañía salió a bolsa mediante una oferta pública inicial (IPO) en 2004. En 2015, Google se reorganizó como una subsidiaria de propiedad absoluta de Alphabet Inc. Google es la subsidiaria más grande de Alphabet y un holding para las propiedades e intereses de internet de Alphabet. Sundar Pichai fue nombrado CEO de Google el 24 de octubre de 2015, en reemplazo de Larry Page, quien se convirtió en el CEO de Alphabet. El 3 de diciembre de 2019, Pichai también se convirtió en el CEO de Alphabet."
print(google_text)

Google fue fundada el 4 de septiembre de 1998 por los informáticos Larry Page y Sergey Brin mientras estudiaban doctorado en la Universidad de Stanford, California. Juntos, poseen aproximadamente el 14 % de sus acciones cotizadas en bolsa y controlan el 56 % del poder de voto de sus accionistas mediante acciones con supervoto. La compañía salió a bolsa mediante una oferta pública inicial (IPO) en 2004. En 2015, Google se reorganizó como una subsidiaria de propiedad absoluta de Alphabet Inc. Google es la subsidiaria más grande de Alphabet y un holding para las propiedades e intereses de internet de Alphabet. Sundar Pichai fue nombrado CEO de Google el 24 de octubre de 2015, en reemplazo de Larry Page, quien se convirtió en el CEO de Alphabet. El 3 de diciembre de 2019, Pichai también se convirtió en el CEO de Alphabet.


In [14]:
spacy_doc_1 = nlp(google_text)

In [15]:
for word in spacy_doc_1.ents:
    print(word.text, word.label_)

el 4 de septiembre de 1998 ORG
Larry Page PERSON
Sergey Brin PERSON
Universidad de Stanford FAC
California GPE
14 % PERCENT
56 % PERCENT
La PERSON
una oferta pública inicial ORG
IPO ORG
2004 DATE
2015 DATE
Google ORG
subsidiaria de propiedad absoluta de Alphabet Inc. PERSON
de Alphabet PERSON
un ORG
para las propiedades e PRODUCT
Sundar Pichai PERSON
24 CARDINAL
Larry Page PERSON
quien se convirtió en PERSON
el CEO de Alphabet ORG
Pichai GPE
el CEO de Alphabet ORG


In [16]:
displacy.render(spacy_doc_1,style="ent", jupyter= True)

In [17]:
#eliminar puntuación y minúsculas
google_text_clean = re.sub(r"[^\w\s]", "",google_text).lower()
print(google_text_clean)

google fue fundada el 4 de septiembre de 1998 por los informáticos larry page y sergey brin mientras estudiaban doctorado en la universidad de stanford california juntos poseen aproximadamente el 14  de sus acciones cotizadas en bolsa y controlan el 56  del poder de voto de sus accionistas mediante acciones con supervoto la compañía salió a bolsa mediante una oferta pública inicial ipo en 2004 en 2015 google se reorganizó como una subsidiaria de propiedad absoluta de alphabet inc google es la subsidiaria más grande de alphabet y un holding para las propiedades e intereses de internet de alphabet sundar pichai fue nombrado ceo de google el 24 de octubre de 2015 en reemplazo de larry page quien se convirtió en el ceo de alphabet el 3 de diciembre de 2019 pichai también se convirtió en el ceo de alphabet


In [18]:
spacy_doc_1_clean = nlp(google_text_clean)

In [19]:
for word in spacy_doc_1_clean.ents:
    print(word.text, word.label_)

google fue PRODUCT
el 4 de septiembre de 1998 ORG
los informáticos NORP
larry PERSON
la universidad de stanford ORG
el 14  de sus ORG
56 CARDINAL
la compañía PERSON
una oferta pública ORG
subsidiaria de propiedad absoluta de alphabet inc google PERSON
la GPE
un ORG
para las propiedades e PRODUCT
pichai fue nombrado ORG
de google PERSON
quien se convirtió en PERSON
el ceo de alphabet el 3 de diciembre de 2019 ORG
el ceo de alphabet ORG


In [20]:
displacy.render(spacy_doc_1_clean, style="ent", jupyter= True)